In [5]:
import os
import subprocess
import glob

# 사용자 설정: 이 부분을 반드시 수정해주세요!
GIT_USER_NAME = "Your GitHub Username" # 예: "checkmate-coding"
GIT_USER_EMAIL = "your.email@example.com" # 예: "your.email@example.com"
GITHUB_REPO_URL = "https://github.com/checkmate-coding/practice.git" # 사용자 GitHub 레포지토리 URL

# --- Git 설정 시작 ---
print("Git 설정을 시작합니다...")

# 1. 사용자 이름과 이메일 설정 확인 및 적용
if GIT_USER_NAME == "checkmate-coding" or GIT_USER_EMAIL == "jaehyun0111@kyonggi.ac.kr":
    print("🛑 경고: GIT_USER_NAME 또는 GIT_USER_EMAIL을 실제 GitHub 정보로 업데이트해야 합니다.")
    print("설정을 업데이트한 후 다시 실행해주세요. 현재 설정으로 계속하면 인증 오류가 발생할 수 있습니다.")

!git config --global user.email "{GIT_USER_EMAIL}"
!git config --global user.name "{GIT_USER_NAME}"

# 2. Git 저장소 초기화 또는 확인
if not os.path.exists(".git"):
    print("Git 저장소를 초기화합니다...")
    !git init
    # `git init`은 보통 `master` 브랜치를 생성합니다. 이를 `main`으로 변경합니다.
    print("기본 브랜치를 'main'으로 설정합니다...")
    !git branch -M main # Rename current branch (master) to main
    print("로컬 브랜치를 'main'으로 전환합니다.")
    !git checkout main
else:
    print("이미 Git 저장소입니다. 'main' 브랜치로 전환합니다.")
    try:
        # Check if 'main' exists, if not, try 'master' or create 'main'
        subprocess.run(["git", "checkout", "main"], check=True, capture_output=True)
    except subprocess.CalledProcessError:
        print("'main' 브랜치가 존재하지 않습니다. 'master' 브랜치 확인 후 전환 시도 또는 'main' 생성합니다.")
        try:
            subprocess.run(["git", "checkout", "master"], check=True, capture_output=True)
            print("'master' 브랜치로 전환했습니다. 'main'으로 변경합니다.")
            !git branch -M main
        except subprocess.CalledProcessError:
            print("로컬에 'main' 또는 'master' 브랜치가 없습니다. 새 'main' 브랜치를 생성합니다.")
            !git checkout -b main

# 3. 노트북 파일 찾기 및 Git에 추가
print("\n🚨 중요: 이 셀을 실행하기 전에 현재 Colab 노트북을 반드시 저장해주세요 (파일 > 저장 또는 Ctrl+S).")
print("저장된 노트북 파일을 Git에 추가합니다...")

# 현재 디렉토리에서 .ipynb 파일들을 찾습니다. (저장되어 있다면 /content/ 에 있을 것입니다)
notebook_files_in_content = glob.glob("/content/*.ipynb")
if not notebook_files_in_content:
    print("경고: '/content/' 디렉토리에서 '.ipynb' 파일을 찾을 수 없습니다. 노트북이 저장되었는지 확인해주세요.")
    print("그래도 'git add .' 명령으로 모든 변경사항을 추가하려고 시도합니다.")
else:
    # 가장 최근에 수정된 .ipynb 파일 (현재 노트북일 가능성이 높음)을 추가합니다.
    notebook_files_in_content.sort(key=os.path.getmtime, reverse=True)
    current_notebook_filename = os.path.basename(notebook_files_in_content[0])
    print(f"발견된 노트북 파일: '{current_notebook_filename}'을(를) Git에 추가합니다.")
    !git add "{current_notebook_filename}"

# 기타 변경사항 (예: .config 디렉토리, sample_data 등 Colab 환경 파일)을 추가합니다.
print("기타 변경사항을 Git에 추가합니다...")
!git add .

# 4. 변경사항 커밋
COMMIT_MESSAGE = "Colab 노트북 파일 및 환경 초기 커밋"
print(f"\n변경사항을 커밋합니다: '{COMMIT_MESSAGE}'")

# 커밋할 변경사항이 있는지 확인합니다.
commit_status = subprocess.run(["git", "status", "--porcelain"], capture_output=True, text=True)
if commit_status.stdout:
    !git commit -m "{COMMIT_MESSAGE}"
else:
    print("커밋할 변경사항이 없습니다.")
    print("새로운 파일을 추가하거나 기존 파일을 수정하지 않았다면 이 메시지가 나타날 수 있습니다.")

# 5. 원격 저장소(remote) 설정
print(f"\n원격 저장소 '{GITHUB_REPO_URL}'를 설정합니다...")
try:
    subprocess.run(["git", "remote", "add", "origin", GITHUB_REPO_URL], check=True, capture_output=True)
    print("원격 'origin'이 추가되었습니다.")
except subprocess.CalledProcessError as e:
    if "remote origin already exists" in e.stderr.decode():
        print("원격 'origin'이 이미 존재합니다. URL을 업데이트합니다.")
        !git remote set-url origin "{GITHUB_REPO_URL}"
    else:
        print(f"원격 'origin' 설정 중 오류 발생: {e.stderr.decode()}")
        # Fallback for debugging, if stderr is not clear
        !git remote -v
        raise

# 6. GitHub에 푸쉬 (인증 필요)
print("\nGitHub에 푸쉬를 시도합니다.")
print("🚨🚨🚨 중요: GitHub에 푸쉬하려면 인증이 필요합니다. 🚨🚨🚨")
print("  개인 액세스 토큰 (Personal Access Token)을 사용하여 푸쉬해야 합니다.")
print("  개인 액세스 토큰은 GitHub 설정에서 발급받을 수 있습니다.")
print("  토큰 사용 방법 예시 (GIT_USER_NAME은 실제 GitHub 사용자 이름이어야 합니다):")
print("  !git push https://<YOUR_PERSONAL_ACCESS_TOKEN>@github.com/{GIT_USER_NAME}/practice.git main")
print("  또는, 이 셀을 실행한 후 나타나는 프롬프트에 사용자 이름과 개인 액세스 토큰을 입력하세요:")
print("  !git push origin main")

# 푸쉬 명령 실행 (로컬 'main' 브랜치를 원격 'main' 브랜치로 푸쉬)
print("\n로컬 'main' 브랜치를 원격 'main' 브랜치로 푸쉬합니다...")
# `-u` 옵션은 upstream(추적 대상)을 설정하여 다음부터는 `git push`만으로도 푸쉬할 수 있게 합니다.
!git push -u origin main

print("\n✅ 스크립트가 완료되었습니다. GitHub 레포지토리에서 푸쉬 성공 여부를 확인해주세요.")
print("푸쉬가 실패했다면, 위에서 안내된 GitHub 인증 절차를 다시 확인하고,")
print("GitHub 레포지토리에 'main' 브랜치가 있는지 또는 보호되어 있지 않은지 확인하세요.")

Git 설정을 시작합니다...
이미 Git 저장소입니다. 'main' 브랜치로 전환합니다.

🚨 중요: 이 셀을 실행하기 전에 현재 Colab 노트북을 반드시 저장해주세요 (파일 > 저장 또는 Ctrl+S).
저장된 노트북 파일을 Git에 추가합니다...
경고: '/content/' 디렉토리에서 '.ipynb' 파일을 찾을 수 없습니다. 노트북이 저장되었는지 확인해주세요.
그래도 'git add .' 명령으로 모든 변경사항을 추가하려고 시도합니다.
기타 변경사항을 Git에 추가합니다...

변경사항을 커밋합니다: 'Colab 노트북 파일 및 환경 초기 커밋'
커밋할 변경사항이 없습니다.
새로운 파일을 추가하거나 기존 파일을 수정하지 않았다면 이 메시지가 나타날 수 있습니다.

원격 저장소 'https://github.com/checkmate-coding/practice.git'를 설정합니다...
원격 'origin'이 이미 존재합니다. URL을 업데이트합니다.

GitHub에 푸쉬를 시도합니다.
🚨🚨🚨 중요: GitHub에 푸쉬하려면 인증이 필요합니다. 🚨🚨🚨
  개인 액세스 토큰 (Personal Access Token)을 사용하여 푸쉬해야 합니다.
  개인 액세스 토큰은 GitHub 설정에서 발급받을 수 있습니다.
  토큰 사용 방법 예시 (GIT_USER_NAME은 실제 GitHub 사용자 이름이어야 합니다):
  !git push https://<YOUR_PERSONAL_ACCESS_TOKEN>@github.com/{GIT_USER_NAME}/practice.git main
  또는, 이 셀을 실행한 후 나타나는 프롬프트에 사용자 이름과 개인 액세스 토큰을 입력하세요:
  !git push origin main

로컬 'main' 브랜치를 원격 'main' 브랜치로 푸쉬합니다...
fatal: could not read Username for 'https://github.com': No such device or address

✅ 스크립트가 완료되었습